## The three communication paths

Every connection a pod makes falls into one of three categories, each routed differently.

**1. Pod-to-pod.** Pod A reaches pod B by IP. The packet leaves A's `eth0`, crosses the `veth` onto the host, and:

- **Same node** — the host bridges directly to B's `veth`. No overlay, no kube-proxy.
- **Different node** — the host routes per the CNI's node routes. Calico BGP = a plain L3 hop; Flannel VXLAN = encapsulated in UDP, sent, unwrapped.

**No NAT** — B sees A's source IP unchanged.

**2. Pod-to-Service.** Pod A calls `http://api:80`. DNS resolves the ClusterIP (notebook 04); `kube-proxy`'s rules **DNAT** it to a random backing pod IP. The destination pod sees the request from **A's IP**, not the Service IP — the Service IP is virtual; only the rule rewrites.

**3. Pod-to-external.** Pod A fetches `https://example.com`. The pod IP isn't public, so the host **source-NATs** to its own IP; return traffic is un-NAT'd. External services see the **node's IP**, not the pod's. To expose the pod IP (rare) you need `externalTrafficPolicy: Local`, an egress gateway, or a mesh.

```
pod→pod: A → B (direct, IP-to-IP)
pod→svc: A → Service(VIP) → kube-proxy DNAT → chosen pod
pod→ext: A → node (source-NAT) → internet
```

These three explain almost every CKA networking question: "why is X unreachable?" becomes "**which path, and which layer is breaking?**" On our map, path 2 runs through the **Service** chip and **kube-proxy**; all three ride the flat fabric under the Networking box.